# 📖 Notebook 1: Order Matching Engine

Robinhood is a **brokerage**, not an exchange. But to understand how orders get filled,
we need to understand what happens on the exchange side. In this notebook we build a
simplified order matching engine so you can see the mechanics first-hand.

## Learning Objectives

By the end of this notebook you will understand:
- The difference between **market orders** and **limit orders**
- How an **order book** works (bids and asks)
- How a matching engine pairs buyers with sellers
- The order lifecycle inside a brokerage: `pending → submitted → filled`
- Why **consistency** matters and how failures are handled

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/robinhood
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `robinhood_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import json
import time
import uuid
from datetime import datetime

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "robinhood_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

# Quick connection test
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM symbols")
    print(f"✅ Connected to PostgreSQL — {cur.fetchone()[0]} symbols loaded")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

## Bad Practice -> Best Practice: Money Should Never Be a `float`

Before we touch the order book, let's see why **every field that represents money in our schema uses `BIGINT` cents, never `NUMERIC` or `FLOAT` dollars.**

Floating-point numbers (IEEE 754) can't represent most decimal fractions exactly. In a brokerage that does millions of trades a day, even tiny rounding errors compound into real, visible money bugs (and regulatory problems).

Run the cell below to see it happen.


In [ ]:
# BAD: prices and quantities as floats (dollars)
bad_price = 0.1 + 0.2          # "should" be 0.3
bad_total = 0.0
for _ in range(1_000_000):
    bad_total += 0.01          # add one cent, a million times -> "should" be $10,000

print("BAD float math")
print(f"   0.1 + 0.2        = {bad_price!r}   (expected 0.3)")
print(f"   0.01 x 1,000,000 = {bad_total!r}   (expected 10000.0)")
print()

# GOOD: prices and quantities as integer cents
good_price_cents = 10 + 20     # 10c + 20c
good_total_cents = 0
for _ in range(1_000_000):
    good_total_cents += 1      # add one cent, a million times

print("GOOD integer cents")
print(f"   10 + 20          = {good_price_cents}   (= 30 cents = $0.30)")
print(f"   1  x 1,000,000   = {good_total_cents}   (= $10,000.00, exactly)")
print()
print("Tip: this is why every *_cents column in db/init.sql is a BIGINT.")
print("     Format to dollars ONLY at the UI edge, e.g. f\"${cents/100:,.2f}\".")


## 📚 Background: How Stock Trading Works

Before we write code, let's understand the key concepts:

### Market Orders vs Limit Orders

| Type | What It Means | Example |
|------|--------------|----------|
| **Market** | "Buy/sell NOW at whatever the current price is" | "Buy 10 shares of AAPL at market price" |
| **Limit**  | "Buy/sell ONLY at this price or better" | "Buy 10 AAPL only if price ≤ $190" |

**Market orders** execute immediately but you don't control the price.  
**Limit orders** give you price control but might never execute if the price never reaches your target.

### The Order Book

An exchange keeps an **order book** — a sorted list of all outstanding limit orders:

```
         ORDER BOOK for AAPL
  ┌──────────────────────────────┐
  │  ASKS (sellers)   ↑ price    │
  │  $192.00  ×  50 shares       │
  │  $191.50  × 100 shares       │  ← lowest ask (best price to buy)
  │ ─────────── SPREAD ───────── │
  │  $191.00  × 200 shares       │  ← highest bid (best price to sell)
  │  $190.50  ×  75 shares       │
  │  BIDS (buyers)    ↓ price    │
  └──────────────────────────────┘
```

The **spread** is the gap between the highest bid and lowest ask.  
When a market buy comes in, it matches against the lowest ask.

## 🏗️ Building a Simple Order Book

Let's build an in-memory order book. This simulates what happens *inside* an exchange.

In [ ]:
import heapq
import itertools
from dataclasses import dataclass, field
from typing import Optional

# A strictly increasing arrival counter. Two orders can land on the same
# wall-clock timestamp (time.time() is microsecond-ish, and much coarser on
# Windows). If price AND timestamp tie, heapq would fall through to comparing
# LimitOrder objects and raise TypeError. `seq` gives every order a unique,
# monotonic arrival rank, so price-time priority is always well defined.
_arrival_seq = itertools.count()


@dataclass
class LimitOrder:
    """A single limit order sitting on the book."""
    order_id: str
    side: str                  # 'buy' or 'sell'
    price_cents: int           # price in cents to avoid float issues
    quantity: int              # shares remaining
    timestamp: float = field(default_factory=time.time)
    seq: int = field(default_factory=lambda: next(_arrival_seq))


class OrderBook:
    """
    A simplified order book for one symbol.

    Bids (buy orders)  are stored in a max-heap (highest price first).
    Asks (sell orders) are stored in a min-heap (lowest price first).

    We use Python's heapq (min-heap) and negate prices for the bid side.
    The heap key is (price, timestamp, seq, order) -- i.e. **price-time
    priority**: best price first, and among equal prices whoever arrived
    first. A partially filled order is pushed back with its ORIGINAL
    timestamp and seq, so it keeps its place at the front of the queue
    instead of going to the back of the line.
    """

    def __init__(self, symbol: str):
        self.symbol = symbol
        self.bids: list = []   # max-heap via negated prices
        self.asks: list = []   # min-heap
        self.trades: list = [] # executed trades log
        # Bookkeeping so we can prove no share is created or destroyed.
        self.submitted: dict = {}  # order_id -> shares submitted
        self.unfilled: dict = {}   # order_id -> shares a market order could NOT fill

    # ---------- read-only views ----------

    def best_bid(self) -> Optional[int]:
        """Highest price anyone is willing to pay, in cents."""
        return -self.bids[0][0] if self.bids else None

    def best_ask(self) -> Optional[int]:
        """Lowest price anyone is willing to sell at, in cents."""
        return self.asks[0][0] if self.asks else None

    def spread_cents(self) -> Optional[int]:
        bid, ask = self.best_bid(), self.best_ask()
        return None if bid is None or ask is None else ask - bid

    # ---------- order entry ----------

    def add_limit_order(self, order: LimitOrder):
        """Place a limit order on the book, matching if possible."""
        self.submitted[order.order_id] = self.submitted.get(order.order_id, 0) + order.quantity
        remaining = order.quantity

        if order.side == 'buy':
            # Try to match against existing asks (sellers), cheapest first
            while remaining > 0 and self.asks:
                best_ask_price, _, _, best_ask = self.asks[0]
                if order.price_cents < best_ask_price:
                    break  # our bid is too low
                # Match! The trade prints at the RESTING order's price --
                # the passive side set the price, the aggressor accepts it.
                heapq.heappop(self.asks)
                matched_qty = min(remaining, best_ask.quantity)
                self._record_trade(order.order_id, best_ask.order_id, best_ask_price, matched_qty)
                remaining -= matched_qty
                best_ask.quantity -= matched_qty
                if best_ask.quantity > 0:
                    # Same price/timestamp/seq -> the residual keeps its queue slot
                    heapq.heappush(self.asks,
                                   (best_ask.price_cents, best_ask.timestamp, best_ask.seq, best_ask))

            # If shares remain, rest on the book
            if remaining > 0:
                order.quantity = remaining
                heapq.heappush(self.bids, (-order.price_cents, order.timestamp, order.seq, order))

        else:  # sell
            while remaining > 0 and self.bids:
                neg_best_bid, _, _, best_bid = self.bids[0]
                best_bid_price = -neg_best_bid
                if order.price_cents > best_bid_price:
                    break  # our ask is too high
                heapq.heappop(self.bids)
                matched_qty = min(remaining, best_bid.quantity)
                self._record_trade(best_bid.order_id, order.order_id, best_bid_price, matched_qty)
                remaining -= matched_qty
                best_bid.quantity -= matched_qty
                if best_bid.quantity > 0:
                    heapq.heappush(self.bids,
                                   (-best_bid.price_cents, best_bid.timestamp, best_bid.seq, best_bid))

            if remaining > 0:
                order.quantity = remaining
                heapq.heappush(self.asks, (order.price_cents, order.timestamp, order.seq, order))

        self.check_invariants()

    def add_market_order(self, side: str, quantity: int, order_id: str = "MKT") -> list:
        """
        Execute a market order — matches immediately at best available price.

        A market order never rests on the book: whatever the book cannot fill
        is cancelled, not queued. We remember that residual in self.unfilled
        so the accounting still balances.
        """
        self.submitted[order_id] = self.submitted.get(order_id, 0) + quantity
        fills = []
        remaining = quantity

        if side == 'buy':
            while remaining > 0 and self.asks:
                best_price, _, _, best_ask = heapq.heappop(self.asks)
                matched_qty = min(remaining, best_ask.quantity)
                fills.append((best_price, matched_qty))
                # A fill IS an execution, so it has to land in the trade log.
                # Returning fills without recording the trade would leave the
                # resting seller's side of the deal written down nowhere.
                self._record_trade(order_id, best_ask.order_id, best_price, matched_qty)
                remaining -= matched_qty
                best_ask.quantity -= matched_qty
                if best_ask.quantity > 0:
                    heapq.heappush(self.asks,
                                   (best_ask.price_cents, best_ask.timestamp, best_ask.seq, best_ask))
        else:
            while remaining > 0 and self.bids:
                neg_price, _, _, best_bid = heapq.heappop(self.bids)
                matched_qty = min(remaining, best_bid.quantity)
                fills.append((-neg_price, matched_qty))
                self._record_trade(best_bid.order_id, order_id, -neg_price, matched_qty)
                remaining -= matched_qty
                best_bid.quantity -= matched_qty
                if best_bid.quantity > 0:
                    heapq.heappush(self.bids,
                                   (-best_bid.price_cents, best_bid.timestamp, best_bid.seq, best_bid))

        if remaining > 0:
            self.unfilled[order_id] = self.unfilled.get(order_id, 0) + remaining
            print(f"⚠️  Market order {order_id}: {remaining} share(s) unfilled — "
                  f"the book ran out of liquidity (market orders never rest).")

        self.check_invariants()
        return fills

    def _record_trade(self, buyer_id: str, seller_id: str, price_cents: int, quantity: int):
        self.trades.append({
            "buyer": buyer_id,
            "seller": seller_id,
            "price_cents": price_cents,
            "quantity": quantity,
            "time": time.time()
        })

    # ---------- the three things that must always be true ----------

    def check_invariants(self):
        """
        Run after every order. If any of these stop holding, the book is
        lying about money and we want to know immediately, not later.
        """
        # 1. The book is never crossed: the spread cannot go negative.
        #    Anything that crosses must have traded instead of resting.
        spread = self.spread_cents()
        assert spread is None or spread > 0, (
            f"crossed book: best ask {self.best_ask()} <= best bid {self.best_bid()}"
        )

        # 2. Every trade has exactly one buyer and one seller of the same size,
        #    so the two sides must sum to the same total.
        bought, sold = {}, {}
        for t in self.trades:
            bought[t["buyer"]] = bought.get(t["buyer"], 0) + t["quantity"]
            sold[t["seller"]] = sold.get(t["seller"], 0) + t["quantity"]
        assert sum(bought.values()) == sum(sold.values()), (
            f"buy side executed {sum(bought.values())} shares but sell side "
            f"executed {sum(sold.values())}"
        )

        # 3. Conservation per order: what we submitted was either executed,
        #    is still resting on the book, or was explicitly left unfilled.
        resting = {}
        for _, _, _, o in self.bids + self.asks:
            resting[o.order_id] = resting.get(o.order_id, 0) + o.quantity
        for oid, submitted in self.submitted.items():
            executed = bought.get(oid, 0) + sold.get(oid, 0)
            accounted = executed + resting.get(oid, 0) + self.unfilled.get(oid, 0)
            assert accounted == submitted, (
                f"order {oid}: submitted {submitted} shares, accounted for {accounted} "
                f"(executed {executed}, resting {resting.get(oid, 0)}, "
                f"unfilled {self.unfilled.get(oid, 0)})"
            )

    def display(self):
        """Pretty-print the current order book."""
        print(f"\n📖 Order Book: {self.symbol}")
        print("=" * 45)

        # Asks (stored lowest first, displayed highest first).
        # Sort on price only — two orders can share a price, and LimitOrder
        # objects are not comparable.
        ask_list = sorted(((p, o) for p, _, _, o in self.asks),
                          key=lambda pair: pair[0], reverse=True)
        print("  ASKS (sellers)")
        if not ask_list:
            print("    (empty)")
        for price, order in ask_list:
            print(f"    ${price/100:>10.2f}  ×  {order.quantity:>4} shares")

        spread = self.spread_cents()
        if spread is None:
            print("  ─────────── SPREAD ───────────")
        else:
            print(f"  ────── SPREAD: ${spread/100:.2f} ──────")

        bid_list = sorted(((-p, o) for p, _, _, o in self.bids),
                          key=lambda pair: pair[0], reverse=True)
        if not bid_list:
            print("    (empty)")
        for price, order in bid_list:
            print(f"    ${price/100:>10.2f}  ×  {order.quantity:>4} shares")
        print("  BIDS (buyers)")
        print()


In [ ]:
# Let's build an order book for AAPL and add some limit orders

book = OrderBook("AAPL")

# Sellers willing to sell at these prices
book.add_limit_order(LimitOrder("S1", "sell", 19200, 50))
book.add_limit_order(LimitOrder("S2", "sell", 19150, 100))
book.add_limit_order(LimitOrder("S3", "sell", 19300, 30))

# Buyers willing to buy at these prices
book.add_limit_order(LimitOrder("B1", "buy", 19100, 200))
book.add_limit_order(LimitOrder("B2", "buy", 19050, 75))
book.add_limit_order(LimitOrder("B3", "buy", 19000, 150))

book.display()

print("💡 Notice the spread: sellers want ≥$191.50, buyers offer ≤$191.00")
print("   No trade happens yet because nobody agrees on a price.")

In [ ]:
# Now a buyer places a limit order at $191.50 — it crosses the spread!

print("🔔 New order: BUY 60 AAPL @ $191.50 (limit)")
print()

book.add_limit_order(LimitOrder("B4", "buy", 19150, 60))

# Show what happened
print("Trades executed:")
for t in book.trades:
    print(f"  {t['buyer']} bought {t['quantity']} shares "
          f"from {t['seller']} @ ${t['price_cents']/100:.2f}")

book.display()

print("💡 The buyer's order matched against S2 (the lowest ask at $191.50).")
print("   S2 had 100 shares, buyer only wanted 60, so 40 shares remain on the ask side.")

In [ ]:
# Now let's try a market order — it takes the best available price

print("🔔 New order: MARKET BUY 50 AAPL")
print()

fills = book.add_market_order("buy", 50, order_id="M1")

total_cost = sum(price * qty for price, qty in fills)
total_shares = sum(qty for _, qty in fills)
avg_price = total_cost / total_shares if total_shares else 0

print("Fills:")
for price, qty in fills:
    print(f"  {qty} shares @ ${price/100:.2f}")

print(f"\nTotal: {total_shares} shares, avg price ${avg_price/100:.2f}")
print(f"Total cost: ${total_cost/100:.2f}")

book.display()

print("💡 The market order ate through the best asks.")
print("   If the book is thin (few shares), a big market order can move the price a lot!")
print("   This is called 'slippage'.")


### ✅ Do the Books Balance?

An order book is an accounting system. Before we move on, assert what the
three cells above *should* have produced — quantities on both sides, the
order the fills came in, and a spread that is still positive.


In [ ]:
# `check_invariants()` already ran after every order; call it once more explicitly
# so it is obvious that it is a real check and not decoration.
book.check_invariants()

# 110 shares changed hands: 60 (B4 vs S2) + 40 (M1 vs S2) + 10 (M1 vs S1).
traded = sum(t["quantity"] for t in book.trades)
assert traded == 110, f"expected 110 shares executed in this scenario, got {traded}"

# Price priority: S2 was the cheapest ask, so all 100 of its shares had to be
# consumed — at its own price of $191.50 — before S1 at $192.00 got a look-in.
assert [(t["seller"], t["quantity"], t["price_cents"]) for t in book.trades] == [
    ("S2", 60, 19150),
    ("S2", 40, 19150),
    ("S1", 10, 19200),
], book.trades

# The market buy paid a blended price because it walked up two levels.
assert avg_price == 19160, f"expected a $191.60 blended fill, got {avg_price}"

# And the book is still sane: bid below ask, spread strictly positive.
assert book.best_bid() == 19100 and book.best_ask() == 19200
assert book.spread_cents() == 100

print(f"✅ Books balance: {traded} shares executed on both sides, "
      f"spread ${book.spread_cents()/100:.2f} (never negative).")


## ⏱️ Price-Time Priority (and What a Partial Fill Must Not Do)

The matching rule is **price first, then time**. Two orders at the same price
are served in arrival order, and — this is the part that is easy to get wrong —
an order that is only *partially* filled keeps its place at the **front** of the
queue. Sending the residual to the back of the line would quietly penalise the
trader who got there first.


In [ ]:
# A fresh book so the assertions below stand on their own.
fifo = OrderBook("FIFO-DEMO")

fifo.add_limit_order(LimitOrder("EARLY", "sell", 20000, 100))  # $200.00, arrives first
fifo.add_limit_order(LimitOrder("LATE",  "sell", 20000, 100))  # $200.00, arrives second
fifo.add_limit_order(LimitOrder("CHEAP", "sell", 19900, 10))   # $199.00, arrives last

# 1. Better PRICE beats earlier time — CHEAP arrived last but fills first.
fifo.add_market_order("buy", 10, order_id="MKT-A")
assert fifo.trades[-1]["seller"] == "CHEAP", fifo.trades

# 2. At equal price, the EARLIER order fills first. This takes 40 of EARLY's
#    100 shares, leaving a 60-share residual.
fifo.add_market_order("buy", 40, order_id="MKT-B")
assert fifo.trades[-1]["seller"] == "EARLY"
resting = {o.order_id: o.quantity for _, _, _, o in fifo.asks}
assert resting == {"EARLY": 60, "LATE": 100}, resting

# 3. The residual keeps its slot: the next 60 shares come from EARLY, not LATE.
fifo.add_market_order("buy", 60, order_id="MKT-C")
assert [t["seller"] for t in fifo.trades] == ["CHEAP", "EARLY", "EARLY"], fifo.trades
resting = {o.order_id: o.quantity for _, _, _, o in fifo.asks}
assert resting == {"LATE": 100}, f"LATE should still be untouched, got {resting}"

# 4. A market order bigger than the book does NOT rest — it is left unfilled.
fifo.add_market_order("buy", 500, order_id="MKT-D")
assert fifo.unfilled["MKT-D"] == 400, fifo.unfilled
assert not fifo.asks, "the ask side should be empty after MKT-D swept it"

print("✅ Price-time priority holds: best price first, then FIFO, and a partial")
print("   fill keeps its queue position instead of going to the back of the line.")


## 🏦 Brokerage Order Lifecycle

Now let's see the **brokerage side** — how Robinhood handles an order from the user
through to the exchange and back.

```
  User App          Robinhood Backend         Exchange
  ───────          ─────────────────         ────────
    │                     │                      │
    │  POST /order        │                      │
    │ ──────────────────► │                      │
    │                     │  1. Save order        │
    │                     │     status=pending    │
    │                     │                      │
    │                     │  2. Submit to         │
    │                     │     exchange ────────►│
    │                     │                      │
    │                     │  3. Get external ID  │
    │                     │  ◄───────────────────│
    │                     │     status=submitted  │
    │                     │                      │
    │  {order: submitted} │                      │
    │ ◄────────────────── │                      │
    │                     │                      │
    │                     │  4. Trade feed:       │
    │                     │     order filled     │
    │                     │  ◄───────────────────│
    │                     │     status=filled     │
    │                     │                      │
    │  {order: filled}    │                      │
    │ ◄────────────────── │                      │
```

**Key insight**: We save to our database *before* talking to the exchange.  
This way, if anything fails mid-way, we have a record and can recover.

In [ ]:
# Simulate the full brokerage order lifecycle against our Postgres database

import random

def simulate_exchange_submit(order_id, symbol, side, qty, price_cents):
    """Pretend to talk to an exchange. Returns an external order ID."""
    time.sleep(0.05)  # simulate network latency
    return f"EX-{uuid.uuid4().hex[:8]}"

def create_order(user_id: int, ticker: str, side: str, order_type: str,
                 quantity: int, limit_price_cents: int = None):
    """
    Full brokerage order flow:
    1. Save order as 'pending'
    2. Submit to exchange
    3. Update to 'submitted' with external ID
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Look up the symbol
    cur.execute("SELECT id, last_price_cents FROM symbols WHERE ticker = %s", (ticker,))
    symbol = cur.fetchone()
    if not symbol:
        print(f"❌ Unknown ticker: {ticker}")
        conn.close()
        return None

    # Step 1: Save order as PENDING
    cur.execute("""
        INSERT INTO orders (user_id, symbol_id, side, order_type, quantity, limit_price_cents, status)
        VALUES (%s, %s, %s, %s, %s, %s, 'pending')
        RETURNING id, status, created_at
    """, (user_id, symbol['id'], side, order_type, quantity, limit_price_cents))
    order = cur.fetchone()
    print(f"📝 Step 1: Order #{order['id']} saved as PENDING")

    # Step 2: Submit to exchange
    price = limit_price_cents or symbol['last_price_cents']
    try:
        ext_id = simulate_exchange_submit(order['id'], ticker, side, quantity, price)
        print(f"📡 Step 2: Submitted to exchange → external ID: {ext_id}")
    except Exception as e:
        # If exchange fails, mark order as failed
        cur.execute("UPDATE orders SET status = 'failed' WHERE id = %s", (order['id'],))
        print(f"❌ Step 2: Exchange submission failed: {e}")
        conn.close()
        return None

    # Step 3: Update order with external ID and status
    cur.execute("""
        UPDATE orders SET status = 'submitted', external_order_id = %s, updated_at = NOW()
        WHERE id = %s
        RETURNING *
    """, (ext_id, order['id']))
    updated = cur.fetchone()
    print(f"✅ Step 3: Order #{updated['id']} updated to SUBMITTED")

    conn.close()
    return dict(updated)


# Create a market buy order for Alice (user 1)
print("=" * 60)
print("Alice wants to BUY 10 shares of GOOGL at market price")
print("=" * 60)
order = create_order(user_id=1, ticker="GOOGL", side="buy",
                     order_type="market", quantity=10)
print()
print(f"Order result: {order['side'].upper()} {order['quantity']} shares, "
      f"status={order['status']}")

In [ ]:
# Now simulate the exchange filling the order (trade feed callback)

def simulate_trade_fill(order_id: int, fill_price_cents: int):
    """
    Simulate receiving a fill from the exchange's trade feed.

    Two things matter here, and both are easy to get wrong:
      * the trade row and the status change are ONE transaction. An order
        marked 'filled' with no matching `trades` row is a hole in the ledger —
        the position can never be rebuilt from history.
      * trade feeds are at-least-once, so a replayed fill must be a no-op.
    """
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = False          # both writes, or neither
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Lock the order row so a concurrent replay of the same fill waits here
        cur.execute("SELECT * FROM orders WHERE id = %s FOR UPDATE", (order_id,))
        order = cur.fetchone()

        if not order:
            print(f"❌ Order #{order_id} not found")
            conn.rollback()
            return None

        if order['status'] == 'filled':
            print(f"↩️  Order #{order_id} already filled — duplicate fill ignored")
            conn.rollback()
            return dict(order)

        # Record the trade (the ledger entry) ...
        cur.execute("""
            INSERT INTO trades (order_id, symbol_id, price_cents, quantity)
            VALUES (%s, %s, %s, %s)
        """, (order['id'], order['symbol_id'], fill_price_cents, order['quantity']))

        # ... and the status change, in the SAME transaction.
        cur.execute("""
            UPDATE orders
            SET status = 'filled',
                filled_quantity = quantity,
                filled_avg_price_cents = %s,
                updated_at = NOW()
            WHERE id = %s
            RETURNING *
        """, (fill_price_cents, order['id']))
        updated = cur.fetchone()
        conn.commit()

        print(f"✅ Order #{updated['id']} FILLED: {updated['filled_quantity']} shares "
              f"@ ${fill_price_cents/100:.2f}")
        return dict(updated)
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


# Simulate the fill
print("📡 Exchange trade feed: order filled!")
print()
filled = simulate_trade_fill(order['id'], fill_price_cents=17830)

# The feed is at-least-once — the same fill can arrive twice.
simulate_trade_fill(order['id'], fill_price_cents=17830)

# The ledger has to back up the order, exactly once.
_conn = get_db()
_cur = _conn.cursor()
_cur.execute("SELECT COUNT(*), COALESCE(SUM(quantity), 0) FROM trades WHERE order_id = %s",
             (order['id'],))
trade_count, ledger_qty = _cur.fetchone()
_conn.close()
assert trade_count == 1, f"the duplicate fill created {trade_count} trade rows, expected 1"
assert ledger_qty == filled['filled_quantity'], (
    f"order says {filled['filled_quantity']} shares filled but the trades table "
    f"sums to {ledger_qty} — a fill must always be backed by a ledger entry"
)
print(f"\n📚 Ledger check: {trade_count} trade row, {ledger_qty} shares "
      f"= order.filled_quantity ✅")


In [ ]:
# View all orders for Alice

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT o.id, s.ticker, o.side, o.order_type, o.quantity,
           o.status, o.filled_quantity,
           o.filled_avg_price_cents / 100.0 AS filled_price,
           o.created_at
    FROM orders o
    JOIN symbols s ON o.symbol_id = s.id
    WHERE o.user_id = 1
    ORDER BY o.created_at DESC
""")

print("📋 Alice's Orders")
print("=" * 90)
print(f"{'ID':>4} {'Ticker':<6} {'Side':<5} {'Type':<7} {'Qty':>5} {'Status':<12} "
      f"{'Filled':>6} {'Price':>10}")
print("-" * 90)

for row in cur.fetchall():
    price_str = f"${row['filled_price']:.2f}" if row['filled_price'] else "-"
    print(f"{row['id']:>4} {row['ticker']:<6} {row['side']:<5} {row['order_type']:<7} "
          f"{row['quantity']:>5} {row['status']:<12} {row['filled_quantity']:>6} {price_str:>10}")

conn.close()

## Bad Practice -> Best Practice: Idempotency Keys for Order Placement

What if the mobile app sends `POST /orders`, the network times out, and the user taps **Buy** again?
Without protection we'd place **two** orders -- the user wanted 10 shares of AAPL and got 20.

The fix is an **idempotency key** (often called a `client_order_id`): the client generates a UUID, sends it with the order, and the server de-duplicates. Retrying the same request returns the *same* order, never a duplicate.


In [ ]:
# BAD: no idempotency -- a retry creates a duplicate order
def bad_place_order(user_id, ticker, quantity):
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("SELECT id FROM symbols WHERE ticker=%s", (ticker,))
    symbol_id = cur.fetchone()["id"]
    cur.execute("""
        INSERT INTO orders (user_id, symbol_id, side, order_type, quantity, status)
        VALUES (%s, %s, 'buy', 'market', %s, 'pending') RETURNING id
    """, (user_id, symbol_id, quantity))
    oid = cur.fetchone()["id"]
    conn.close()
    return oid

# Simulate the client retrying the same "Buy 5 AAPL" click twice
o1 = bad_place_order(3, "AAPL", 5)
o2 = bad_place_order(3, "AAPL", 5)   # retry!
print(f"BAD flow created TWO orders: #{o1} and #{o2} -- the user is double-charged")
print()


# GOOD: use a client-generated idempotency key and a UNIQUE index
conn = get_db()
cur = conn.cursor()
# For this demo we reuse the external_order_id column as the idempotency key
cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS ux_orders_client_key ON orders(external_order_id)")
conn.close()

def good_place_order(user_id, ticker, quantity, client_order_id):
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    # Already seen this client_order_id? Return the existing order instead of inserting a new one.
    cur.execute("SELECT id FROM orders WHERE external_order_id = %s", (client_order_id,))
    row = cur.fetchone()
    if row:
        conn.close()
        return row["id"], "replayed"
    cur.execute("SELECT id FROM symbols WHERE ticker=%s", (ticker,))
    symbol_id = cur.fetchone()["id"]
    try:
        cur.execute("""
            INSERT INTO orders (user_id, symbol_id, side, order_type, quantity, status, external_order_id)
            VALUES (%s, %s, 'buy', 'market', %s, 'pending', %s) RETURNING id
        """, (user_id, symbol_id, quantity, client_order_id))
        oid = cur.fetchone()["id"]
        conn.close()
        return oid, "created"
    except psycopg2.errors.UniqueViolation:
        # Lost a race with a concurrent retry -- fetch the winner
        conn.rollback()
        cur.execute("SELECT id FROM orders WHERE external_order_id = %s", (client_order_id,))
        oid = cur.fetchone()["id"]
        conn.close()
        return oid, "replayed"

key = f"cli-{uuid.uuid4()}"           # the phone generates ONE key per Buy tap
a, s1 = good_place_order(3, "AAPL", 5, key)
b, s2 = good_place_order(3, "AAPL", 5, key)   # retry with same key
print(f"GOOD flow: first call -> order #{a} ({s1}), retry -> order #{b} ({s2})")
assert a == b, "Retry must return the SAME order id"
print("Tip: same key => same order id. No duplicates, even with flaky networks.")


## 🛡️ Handling Failures: Why Order Consistency Matters

What happens if the system crashes *between* submitting to the exchange and updating our DB?

```
  Failure Points in the Order Flow:
  
  1. Save order (pending)    ← Failure here? Easy: tell user it failed.
  2. Submit to exchange       ← Failure here? Mark as failed, done.
  3. Update DB (submitted)    ← Failure here? 😱 Exchange has our order
                                 but our DB still says 'pending'!
```

**Solution**: A cleanup job scans for orders stuck in `pending` status and checks
the exchange to see if they actually went through. Let's simulate this.

In [ ]:
# Simulate a failure: orders submitted to the exchange but the DB update crashed

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Two orders that "got stuck" in pending five minutes ago.
cur.execute("""
    INSERT INTO orders (user_id, symbol_id, side, order_type, quantity, status, created_at)
    VALUES (2, 1, 'buy', 'market', 5, 'pending', NOW() - INTERVAL '5 minutes'),
           (2, 1, 'buy', 'market', 7, 'pending', NOW() - INTERVAL '5 minutes')
    RETURNING id
""")
landed_id, lost_id = [row['id'] for row in cur.fetchall()]
conn.close()

print(f"⚠️  Two orders stuck in 'pending': #{landed_id} and #{lost_id}")
print("    Our database cannot tell them apart — only the exchange knows which one landed.")
print()

# Stand-in for the exchange's "look up my order by client order id" API.
# No coin flip: one of these two really did reach the exchange before we
# crashed, and the other never left the building. That is the whole reason
# the reconciler has to *ask* rather than guess.
EXCHANGE_ORDERS = {landed_id: {"status": "filled", "price_cents": 19000}}


def query_exchange(order_id: int):
    """What the exchange says about an order we are not sure about."""
    return EXCHANGE_ORDERS.get(order_id)


def cleanup_stuck_orders(max_age_minutes: int = 2):
    """
    Background job that finds orders stuck in 'pending' for too long and asks
    the exchange what actually happened to each one.
    """
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = False          # reconciling a fill is a money write
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("""
        SELECT id, user_id, symbol_id, side, quantity
        FROM orders
        WHERE status = 'pending'
          AND created_at < NOW() - (%s::text || ' minutes')::interval
        ORDER BY id
    """, (max_age_minutes,))

    stuck = cur.fetchall()
    print(f"🔍 Cleanup job found {len(stuck)} stuck order(s)")

    for order in stuck:
        remote = query_exchange(order['id'])

        if remote and remote['status'] == 'filled':
            # Reconciling to 'filled' writes the trade row too, in the same
            # transaction. Flipping the status on its own would leave an order
            # that claims a fill with no execution behind it — the position and
            # P&L history could never be made to add up again.
            cur.execute("""
                INSERT INTO trades (order_id, symbol_id, price_cents, quantity)
                VALUES (%s, %s, %s, %s)
            """, (order['id'], order['symbol_id'], remote['price_cents'], order['quantity']))
            cur.execute("""
                UPDATE orders SET status = 'filled', filled_quantity = quantity,
                       filled_avg_price_cents = %s, updated_at = NOW()
                WHERE id = %s
            """, (remote['price_cents'], order['id']))
            print(f"  ✅ Order #{order['id']} was filled on the exchange → "
                  f"marked filled AND trade recorded")
        else:
            cur.execute("""
                UPDATE orders SET status = 'failed', updated_at = NOW()
                WHERE id = %s
            """, (order['id'],))
            print(f"  ❌ Order #{order['id']} never reached the exchange → marked failed")

    conn.commit()
    conn.close()

cleanup_stuck_orders()

# The reconciler must resolve both orders, and must never leave one claiming a
# fill it cannot back up with trade rows.
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT o.id, o.status, o.filled_quantity, COALESCE(SUM(t.quantity), 0) AS ledger_qty
    FROM orders o
    LEFT JOIN trades t ON t.order_id = o.id
    WHERE o.id IN (%s, %s)
    GROUP BY o.id, o.status, o.filled_quantity
""", (landed_id, lost_id))
resolved = {row[0]: row[1:] for row in cur.fetchall()}
conn.close()

assert resolved[landed_id] == ('filled', 5, 5), (
    f"the order the exchange filled should be 'filled' for 5 shares with 5 shares "
    f"in the ledger, got {resolved[landed_id]}"
)
assert resolved[lost_id] == ('failed', 0, 0), (
    f"the order the exchange never saw should be 'failed' with nothing in the "
    f"ledger, got {resolved[lost_id]}"
)

print()
print(f"📚 #{landed_id} → filled, {resolved[landed_id][2]} shares in the ledger.")
print(f"📚 #{lost_id} → failed, nothing in the ledger. ✅")
print("💡 This cleanup pattern ensures eventual consistency.")
print("   Even if our system crashes, no order is lost or duplicated — and no")
print("   order is marked filled without the matching trade row.")


## 🔄 Cancel Flow

Cancelling an order follows a similar pattern with its own safety steps:

1. Mark order as `pending_cancel` (so cleanup jobs know what we intended)
2. Send cancel request to exchange
3. Update order to `cancelled`

If step 3 fails, the cleanup job picks up `pending_cancel` orders and verifies with the exchange.

In [ ]:
# Create a limit order, then cancel it

print("1️⃣  Create a limit order")
limit_order = create_order(user_id=1, ticker="TSLA", side="buy",
                           order_type="limit", quantity=25,
                           limit_price_cents=24000)
print()

def cancel_order(order_id: int):
    """Cancel an outstanding order."""
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Step 1: Mark as pending_cancel
    cur.execute("""
        UPDATE orders SET status = 'pending_cancel', updated_at = NOW()
        WHERE id = %s AND status IN ('pending', 'submitted')
        RETURNING *
    """, (order_id,))
    order = cur.fetchone()

    if not order:
        print(f"❌ Order #{order_id} cannot be cancelled (already filled or cancelled)")
        conn.close()
        return None

    print(f"📝 Step 1: Order #{order_id} marked as PENDING_CANCEL")

    # Step 2: Send cancel to exchange
    time.sleep(0.03)  # simulate network
    print(f"📡 Step 2: Cancel sent to exchange")

    # Step 3: Update status
    cur.execute("""
        UPDATE orders SET status = 'cancelled', updated_at = NOW()
        WHERE id = %s RETURNING *
    """, (order_id,))
    cancelled = cur.fetchone()
    print(f"✅ Step 3: Order #{order_id} CANCELLED")

    conn.close()
    return dict(cancelled)

print("2️⃣  Cancel the order")
cancel_order(limit_order['id'])

## 🧹 Cleanup

In [ ]:
# Remove the test orders we created during this notebook
conn = get_db()
cur = conn.cursor()
cur.execute("DROP INDEX IF EXISTS ux_orders_client_key")
cur.execute("DELETE FROM trades WHERE order_id > 5")
cur.execute("DELETE FROM orders WHERE id > 5")
print("Cleaned up test orders, trades, and the demo idempotency index")
conn.close()


## 📚 Summary

### Key Takeaways

1. **Order Book** — a sorted list of bids (buy) and asks (sell). Matching happens when prices cross.
2. **Price-time priority** — best price first, then earliest arrival. A partially filled order keeps
   its place at the front of the queue; it does not go to the back of the line.
3. **Market orders** execute immediately at the best available price; **limit orders** wait for a target price.
   A market order never rests on the book — whatever it cannot fill is cancelled.
4. **The book is never crossed** — after any order is processed, best ask > best bid, so the spread
   is always positive. Anything that would cross must trade instead of resting.
5. **Robinhood is a brokerage**, not an exchange — it routes orders to exchanges and tracks them.
6. **Order lifecycle**: `pending → submitted → filled/cancelled/failed`.
7. **Save to DB before sending to exchange** — this is crucial for crash recovery.
8. **A fill is a ledger entry.** Never flip an order to `filled` without writing the matching
   `trades` row in the *same* transaction — not in the trade-feed handler, and not in the reconciler.
9. **Cleanup jobs** ensure eventual consistency by reconciling stuck orders.
10. **Store money as integer cents** -- floats lose precision and cause real bugs.
11. **Idempotency keys** (`client_order_id`) stop network retries from creating duplicate orders,
    and a replayed *fill* must be a no-op too.

### For System Design Interviews

- Always mention the order status state machine
- Explain why you save to DB first (fault tolerance)
- Mention cleanup/reconciliation jobs for eventual consistency
- State the matching rule as "price, then time" — interviewers listen for the time half
- Use cents (integers) for prices — never floating point for money!

### What This Toy Does NOT Do

Worth saying out loud in an interview, because a real venue does all of it:

- **No self-trade prevention** — nothing stops one participant from matching their own resting order.
- **No stop, stop-limit, IOC/FOK, iceberg or good-till-date orders** — just plain market and limit.
- **No auctions, halts, circuit breakers or tick-size rules.**
- **Single-threaded, single symbol, in memory** — a real engine is a replicated state machine
  with a sequenced input log so every replica reaches the same book.
- **The brokerage half never touches positions or cash** — that is Notebook 2's job.

### Next Up

In **Notebook 2**, we'll track **portfolios and positions** — how buying and selling
updates a user's holdings and profit/loss.
